In [ ]:

from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/data/store-sales-forecasting/train.csv')
df = df.set_index('Date')
df.index = pd.to_datetime(df.index)

In [ ]:
df.columns

In [ ]:
def create_basic_features(data: pd.DataFrame)-> pd.DataFrame:
    data["Day of month"] = data.index.day
    data["Month"] = data.index.month
    data["Year"] = data.index.year
    data["Week of year"] = data.index.isocalendar().week.astype(int)
    data["Day of year"] = data.index.dayofyear
    data["Quarter"] = data.index.quarter
    data["Is weekend"] = (data.index.dayofweek >= 6).astype(int)
    data["StateHoliday"] = data["StateHoliday"].astype(str).astype("category")
    
    return data

TARGET_COL = "Sales"
FEAT_COLS = ["DayOfWeek", "Day of month", "Month", "Year", "Week of year", "Open", "Promo", "StateHoliday", "SchoolHoliday", "Day of year", "Quarter", "Is weekend"]

In [ ]:
df = create_basic_features(df)

## Connecting Mlflow (with dagshub)

In [ ]:
%pip install -q dagshub mlflow

In [ ]:
import dagshub
dagshub.init(repo_owner='PigStep', repo_name='store-sales-forecast', mlflow=True)

In [ ]:
import mlflow

mlflow.autolog()

In [ ]:
#with mlflow.start_run():
  # Your training code here...
#  mlflow.log_metric('accuracy', 42)
#  mlflow.log_param('Param name', 'Value')

## Training baselines

I choose an XGboost model as a baseline for this task

Note: business side want to predict the future with a 6 weeeks horizon.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tss = TimeSeriesSplit(n_splits=5, test_size=24*7*6)
df = df.sort_index()

In [ ]:
%pip install -q xgboost

Submissions are evaluated on the Root Mean Square Percentage Error (RMSPE). The RMSPE is calculated as

$$
\textrm{RMSPE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left(\frac{y_i - \hat{y}_i}{y_i}\right)^2},
$$

where y\_i denotes the sales of a single store on a single day and yhat\_i denotes the corresponding prediction. Any day and store with 0 sales is ignored in scoring.

In [ ]:
def rmspe(y_true, y_pred):
    return float(np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2)))

def rmspe_objective(y_true, y_pred):
    denom = np.square(y_true)
    grad = -2.0 * (y_true - y_pred) / denom
    hess = 2.0 / denom
    return grad, hess

In [ ]:
df.dtypes

In [ ]:
from xgboost import XGBRegressor

for i, (train_index, test_index) in enumerate(tss.split(df)):
    print(10*"=",f"Fold {i}:",10*"=")
    
    train_df = df.iloc[train_index]
    test_df = df.iloc[test_index]
    train_df = train_df[train_df[TARGET_COL] > 0]
    test_df = test_df[test_df[TARGET_COL] > 0]
    
    X_train = train_df[FEAT_COLS]
    y_train = train_df[TARGET_COL]
    
    X_test = test_df[FEAT_COLS]
    y_test = test_df[TARGET_COL]
    
    xgb = XGBRegressor(early_stopping_rounds = 50, enable_categorical = True, objective = rmspe_objective, eval_metric = rmspe)
    
    xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose = 100)
    